# Relational Feature Pruning

This notebook reads the outputs from `scripts/run_relational_feature_pruning.py`. The purpose is to compare grouped one-hot encoding, missing-indicator removal, and gain-based/domain-based pruning without changing the earlier full relational setup.

In [1]:
from pathlib import Path
import pandas as pd

CWD = Path.cwd()
PROJECT_ROOT = CWD if (CWD / 'reports').exists() else CWD.parent
REPORTS_DIR = PROJECT_ROOT / 'reports'
REPORTS_DIR

WindowsPath('C:/Users/erenb/Desktop/Summer2026/DataScience/projects/home-credit-default-risk/reports')

## Validation Comparison

In [2]:
pruning_results = pd.read_csv(REPORTS_DIR / 'relational_feature_pruning_results.csv')
pruning_results.sort_values('validation_roc_auc', ascending=False)

,feature_set,raw_feature_count,transformed_feature_count,numeric_add_indicator,one_hot_min_frequency,one_hot_max_categories,threshold,validation_roc_auc,validation_average_precision,validation_accuracy,validation_precision_class_1,validation_recall_class_1,validation_f1_class_1
0,positive_gain_raw_grouped_ohe,628,1284,True,1000,20,0.669277,0.787927,0.272846,0.856916,0.267646,0.444864,0.334216
1,domain_compact_350_no_missing_indicators,350,399,False,1000,20,0.682093,0.787521,0.272782,0.863380,0.276931,0.429758,0.336819
2,top_450_raw_no_missing_indicators,450,510,False,1000,20,0.657600,0.787442,0.273907,0.850270,0.261285,0.467774,0.335288
3,all_features_no_missing_indicators,908,974,False,1000,20,0.644462,0.787375,0.272529,0.842852,0.253927,0.488419,0.334137
4,positive_gain_raw_no_missing_indicators,628,694,False,1000,20,0.686036,0.787305,0.273519,0.865737,0.279545,0.420443,0.335813
5,top_300_raw_no_missing_indicators,300,349,False,1000,20,0.692981,0.786828,0.272682,0.868501,0.282783,0.409366,0.334499


## Compact Domain Holdout Metrics

In [3]:
compact_metrics = pd.read_csv(REPORTS_DIR / 'relational_domain_compact_final_test_metrics.csv')
compact_metrics

,feature_set,threshold_strategy,threshold,roc_auc,average_precision,accuracy,precision_class_1,recall_class_1,f1_class_1
0,domain_compact_350_no_missing_indicators,default_0.5,0.500000,0.790435,0.289425,0.738354,0.191961,0.698288,0.301138
1,domain_compact_350_no_missing_indicators,validation_selected,0.682093,0.790435,0.289425,0.867193,0.287289,0.435650,0.346246


## Compact Domain Confusion Matrix

In [4]:
pd.read_csv(REPORTS_DIR / 'relational_domain_compact_final_confusion_matrix.csv', index_col=0)

,predicted_0,predicted_1
actual_0,51172,5366
actual_1,2802,2163


## Compact Domain Feature Importance

In [5]:
compact_importance = pd.read_csv(REPORTS_DIR / 'relational_domain_compact_final_feature_importance.csv')
compact_importance.head(50)

,transformed_feature,gain_importance
0,numeric__EXT_SOURCE_MEAN,742010.401367
1,numeric__EXT_SOURCE_MIN,104920.911734
2,numeric__CREDIT_TERM_APPROX,69141.412344
3,numeric__EXT_SOURCE_MAX,67467.253616
4,numeric__BURO_BURO_DEBT_CREDIT_RATIO_MAX,57211.238388
5,numeric__INST_INST_LATE_PAYMENT_FLAG_MEAN,47486.430962
6,numeric__GOODS_CREDIT_RATIO,43679.736755
7,numeric__AGE_YEARS,36348.240257
8,numeric__PREV_DAYS_LAST_DUE_1ST_VERSION_MAX,33564.875463
9,numeric__PREV_PREV_CREDIT_APPLICATION_RATIO_MEAN,29761.266792


## Source Column Gain Mapping

In [6]:
source_gain = pd.read_csv(REPORTS_DIR / 'relational_source_column_gain.csv')
source_gain.head(50)

,source_column,total_gain,max_gain,transformed_feature_count,positive_transformed_feature_count,source_column_exists
0,EXT_SOURCE_MEAN,754268.248363,754268.248363,2,1,True
1,EXT_SOURCE_MIN,111120.871225,111120.871225,2,1,True
2,CREDIT_TERM_APPROX,67951.873411,67951.873411,2,1,True
3,EXT_SOURCE_MAX,59565.386303,59565.386303,2,1,True
4,BURO_BURO_DEBT_CREDIT_RATIO_MAX,56662.530704,56662.530704,2,1,True
5,INST_INST_LATE_PAYMENT_FLAG_MEAN,49238.302052,49238.302052,2,1,True
6,GOODS_CREDIT_RATIO,44594.020910,44594.020910,2,1,True
7,AGE_YEARS,36856.707182,36856.707182,1,1,True
8,PREV_DAYS_LAST_DUE_1ST_VERSION_MAX,32552.885391,32552.885391,2,1,True
9,CODE_GENDER,32340.690775,17534.539440,3,2,True


## Findings

The original full relational model had 908 raw columns and 1884 transformed features. The compact domain model keeps 350 raw columns and produces 399 transformed features by removing most zero/low-gain aggregate columns, grouping rare application one-hot categories, and disabling broad numeric missing-indicator expansion. This smaller model slightly improves holdout ROC-AUC and F1, so it is the better practical baseline for tuning.